<img src="https://full-stack-assets.s3.eu-west-3.amazonaws.com/Walmart_logo_(2008).svg.png" alt="WALMART LOGO" />

# Walmart Predict Weekly Sales 🛒

## Company's Description 📇

Walmart Inc. is an American multinational retail corporation that operates a chain of hypermarkets, discount department stores, and grocery stores from the United States, headquartered in Bentonville, Arkansas. The company was founded by Sam Walton in 1962.

## Project 🚧

Walmart's marketing service has asked you to build a machine learning model able to estimate the weekly sales in their stores, with the best precision possible on the predictions made. Such a model would help them understand better how the sales are influenced by economic indicators, and might be used to plan future marketing campaigns.

## Goals 🎯

The project can be divided into three steps:

- Part 1: make an EDA and all the necessary preprocessings to prepare data for machine learning
- Part 2: train a **linear regression model** (baseline)
- Part 3: avoid overfitting by training a **regularized regression model**

## Scope of this project 🖼️

For this project, you'll work with a dataset that contains information about weekly sales achieved by different Walmart stores, and other variables such as the unemployment rate or the fuel price, that might be useful for predicting the amount of sales. The dataset has been taken from a Kaggle competition, but we made some changes compared to the original data. Please make sure that you're using **our** custom dataset (available on JULIE). 🤓

## Deliverable 📬

To complete this project, your team should: 

- Create some visualizations
- Train at least one **linear regression model** on the dataset, that predicts the amount of weekly sales as a function of the other variables
- Assess the performances of the model by using a metric that is relevant for regression problems
- Interpret the coefficients of the model to identify what features are important for the prediction
- Train at least one model with **regularization (Lasso or Ridge)** to reduce overfitting


___
___

# **PARTIE I - EDA & PREPROCESSING**

### **00 - Imports des bibliotèques**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

df = pd.read_csv("Walmart_Store_sales.csv")

### **01 - Exploration**

In [2]:
df.shape
df.head()
df.tail()
df.info()
df.dtypes
df.columns      

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         150 non-null    float64
 1   Date          132 non-null    object 
 2   Weekly_Sales  136 non-null    float64
 3   Holiday_Flag  138 non-null    float64
 4   Temperature   132 non-null    float64
 5   Fuel_Price    136 non-null    float64
 6   CPI           138 non-null    float64
 7   Unemployment  135 non-null    float64
dtypes: float64(7), object(1)
memory usage: 9.5+ KB


Index(['Store', 'Date', 'Weekly_Sales', 'Holiday_Flag', 'Temperature',
       'Fuel_Price', 'CPI', 'Unemployment'],
      dtype='object')

### **02 - Statistiques descriptives**

In [3]:
df.describe()
df.describe(include="all")

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,150.000000,132,1.360000e+02,138.000000,132.000000,136.000000,138.000000,135.000000
unique,NaN,85,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,19-10-2012,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN
mean,9.866667,NaN,1.249536e+06,0.079710,61.398106,3.320853,179.898509,7.598430
std,6.231191,NaN,6.474630e+05,0.271831,18.378901,0.478149,40.274956,1.577173
min,1.000000,NaN,2.689290e+05,0.000000,18.790000,2.514000,126.111903,5.143000
25%,4.000000,NaN,6.050757e+05,0.000000,45.587500,2.852250,131.970831,6.597500
50%,9.000000,NaN,1.261424e+06,0.000000,62.985000,3.451000,197.908893,7.470000
75%,15.750000,NaN,1.806386e+06,0.000000,76.345000,3.706250,214.934616,8.150000


In [4]:
# Heatmap de corrélation
corr_matrix = df.corr(numeric_only=True)

fig = px.imshow(corr_matrix,
                title="Heatmap de corrélation",
                color_continuous_scale="RdBu_r",
                zmin=-1, zmax=1)
fig.show()

/opt/anaconda3/lib/python3.13/site-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




**Observation :**
La heatmap de corrélation montre que Weekly_Sales n'est corrélée 
à aucune variable économique en particulier (toutes proches de 0). 
Cela suggère que les ventes sont influencées par une combinaison de facteurs 
plutôt que par une seule variable dominante.

### **03 - Valeurs Manquantes**

In [5]:
missing_df = pd.DataFrame({
    "NaN Count" : df.isnull().sum(),
    "NaN %": (df.isnull().sum() / len(df) * 100).round(1)
}).sort_values("NaN %", ascending=False)

missing_df[missing_df["NaN Count"] > 0]

,NaN Count,NaN %
Date,18,12.0
Temperature,18,12.0
Unemployment,15,10.0
Weekly_Sales,14,9.3
Fuel_Price,14,9.3
Holiday_Flag,12,8.0
CPI,12,8.0


##### **Observation:**
des taux assez faibles, pas de supressions de colonnes mais traitement au préprocessing. Sauf Weekly Sales qui est notre variable cible, pour éviter les risques de biais pour le modèle. Ses lignes seront dropé (via **dropna()** )

### **04 - Variable Cible**

In [6]:
df["Weekly_Sales"].value_counts()
df["Weekly_Sales"].value_counts(normalize=True) * 100 

px.histogram(df, x="Weekly_Sales",
        nbins=30,
        title="Distribution de la variable cible : Weekly_Sales")


In [7]:
df["Weekly_Sales"].describe()
df["Weekly_Sales"].skew()

np.float64(0.08147048584829437)

Observations : le SKEW étant normal, le modèle pourra travailler directement avec ces valeurs.

### **05 - NaaN Weekly Sales**

In [8]:
df_clean = df.dropna(subset=["Weekly_Sales"]).copy()
print(df.shape)
print(df_clean.shape)

(150, 8)
(136, 8)


### **06 - Feature Engineering : Date**

In [9]:
##Conversion de Date en format Date
df_clean["Date"] = pd.to_datetime(df_clean["Date"], dayfirst= True)

#Sérparation
df_clean["Year"] = df_clean["Date"].dt.year
df_clean["Month"] = df_clean["Date"].dt.month
df_clean["Day"] = df_clean["Date"].dt.day
df_clean["DayOfWeek"] = df_clean["Date"].dt.dayofweek

df_clean.describe

<bound method NDFrame.describe of      Store       Date  Weekly_Sales  Holiday_Flag  Temperature  Fuel_Price  \
0      6.0 2011-02-18    1572117.54           NaN        59.61       3.045   
1     13.0 2011-03-25    1807545.43           0.0        42.38       3.435   
3     11.0        NaT    1244390.03           0.0        84.57         NaN   
4      6.0 2010-05-28    1644470.66           0.0        78.89       2.759   
5      4.0 2010-05-28    1857533.70           0.0          NaN       2.756   
..     ...        ...           ...           ...          ...         ...   
145   14.0 2010-06-18    2248645.59           0.0        72.62       2.780   
146    7.0        NaT     716388.81           NaN        20.74       2.778   
147   17.0 2010-06-11     845252.21           0.0        57.14       2.841   
148    8.0 2011-08-12     856796.10           0.0        86.05       3.638   
149   19.0 2012-04-20    1255087.26           0.0        55.20       4.170   

            CPI  Unemployment

**Observation:**
on remarque des NAT ( (Not At Time)) dans la colonne de date, et sur ces lignes des NaN dans Year, Month ou Day etc etc. Nous pouvons drop ces lignes, cela permettera au modèle de travailler avec des lignes avec les features completes.

In [10]:
##Drop des lignes 

df_clean = df_clean.dropna(subset=["Date"]).copy()
print(df_clean.shape)

(118, 12)


### **07 - Outliers**

***Nous allons utiliser la règles des 3σ : "statistiquement, dans une distribution normale, 99.7% des données tombent dans cet intervalle. Ce qui est en dehors est vraiment exceptionnel."***

In [11]:
### Temperature
###Définitions du mean et std

temp_mean = df_clean["Temperature"].mean()
temp_std = df_clean["Temperature"].std()

### Filtre par 3σ
df_clean = df_clean[
    (df_clean["Temperature"] >= temp_mean - 3*temp_std) &
    (df_clean["Temperature"] <= temp_mean + 3*temp_std)
]


### Fuel_Price
###Définitions du mean et std

fuelprice_mean = df_clean["Fuel_Price"].mean()
fuelprice_std = df_clean["Fuel_Price"].std()

### Filtre par 3σ
df_clean = df_clean[
    (df_clean["Fuel_Price"] >= fuelprice_mean - 3*fuelprice_std) &
    (df_clean["Fuel_Price"] <= fuelprice_mean + 3*fuelprice_std)
]


### CPI
###Définitions du mean et std

cpi_mean = df_clean["CPI"].mean()
cpi_std = df_clean["CPI"].std()

### Filtre par 3σ
df_clean = df_clean[
    (df_clean["CPI"] >= cpi_mean - 3*cpi_std) &
    (df_clean["CPI"] <= cpi_mean + 3*cpi_std)
]



### Unemployment
###Définitions du mean et std

unemp_mean = df_clean["Unemployment"].mean()
unemp_std = df_clean["Unemployment"].std()

### Filtre par 3σ
df_clean = df_clean[
    (df_clean["Unemployment"] >= unemp_mean - 3*unemp_std) &
    (df_clean["Unemployment"] <= unemp_mean + 3*unemp_std)
]


print(df_clean.shape)

(80, 12)


In [12]:
# Boxplot des features avant suppression des outliers
fig = px.box(df, 
             y=["Temperature", "Fuel_Price", "CPI", "Unemployment"],
             title="Distribution AVANT suppression des outliers")
fig.show()

In [13]:
# Après suppression des outliers
fig = px.box(df_clean, 
             y=["Temperature", "Fuel_Price", "CPI", "Unemployment"],
             title="Distribution APRÈS suppression des outliers")
fig.show()

**Observations :**
Les boxplots avant/après montrent que la suppression des outliers a eu peu d'impact 
visuel sur les distributions — ce qui confirme que les outliers étaient peu nombreux 
mais suffisamment extrêmes pour justifier leur suppression (passage de 118 à 80 lignes).

On note notamment qu'Unemployment avait un point aberrant visible avant nettoyage 
(le point isolé à droite) qui a été supprimé.

### **08 - X et y**

In [14]:
### Définition de y (cible) et X (les features/variable explicatives)

y = df_clean["Weekly_Sales"]
X = df_clean.drop(columns=["Weekly_Sales", "Date"])

### **09 - Train/Test split**

In [15]:
## On va utiliser "train_test_split" de sklearn.

from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

### **10 - Pipeline Sklearn**

**Rappel : OneHotEncoder transforme une variable catégorielle en autant de colonnes binaires qu'il y a de catégories, pour éviter que le modèle interprète un ordre là où il n'y en a pas.**<br>
Ex :<br>
Store<br>
1<br>
2<br>
3<br>
<br>
Après OneHotEncoder : <br>

Store_1 Store_2 Store_3<br>
1       0       0<br>
0       1       0<br>
0       0       1<br>

<br>

***-> Nous allons donc appliquer OneHot encoder (colonnes catégorielles -> colonnes binaires) et StandardScaler pour mettre toutes les features numériques sur la même échelle (équilibrer),pour éviter qu'une feature avec de grandes valeurs (comme CPI 200) écrase une feature avec de petits valeurs (comme Fuel_price 3). <br> <br> -> En scalant après le split, le scaler va apprendre sur X_train uniquement et appliquer sur X_test (données de test qu'il n'aura donc pas vu lors du calcul de moyenne et std).***

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Colonnes à traiter
cat_features = ["Store", "Holiday_Flag"]
num_features = ["Temperature", "Fuel_Price", "CPI", "Unemployment", "Year", "Month", "Day", "DayOfWeek" ]

#Transformers
cat_transformers = Pipeline(steps=[
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
])
num_transformers = Pipeline(steps=[
    ("scaler", StandardScaler())
])

#ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ("cat", cat_transformers, cat_features),
    ("num", num_transformers, num_features)
])

___
___

# **PARTIE II - Linear Regression baseline + évaluation + analyse des coefficients**

### **11 - Entraînement du modèle**

In [17]:
# Connexion au modèle

from sklearn.linear_model import LinearRegression

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

#Entraînement
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Store', 'Holiday_Flag']),
                                                 ('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Temperature', 'Fuel_Price',
                                                   'CPI', 'Unemployment',
                                                   'Year', 'Month', 'Day',
                                                   'DayOfWeek'])])),
                ('regressor', LinearRegression())])

### **12 - Evaluation**

##### **X_Train & X_Test**

In [18]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np 

#TRAIN
#Prédiction sur données de train
y_pred = model.predict(X_train)

#Comparaison des prédicitons avec les vraies valeurs
r2_train = r2_score(y_train, y_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred))


#TEST

#Prédiction sur données de test
y_pred = model.predict(X_test)

#Comparaison des prédicitons avec les vraies valeurs
r2_test = r2_score(y_test, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))


print(f"Train - R²: {r2_train:.4f} | RMSE : {rmse_train:.2f} ")
print(f"Test - R² : {r2_test:.4f} | RMSE : {rmse_test:.2f} ")

Train - R²: 0.9822 | RMSE : 89796.67 
Test - R² : 0.9655 | RMSE : 126960.54 


#### **Observations:**

**-> Nous observons des R2 similaires sur le train et le test, un écart de 0,02. Ce sui laisse prédire qu'il ne semble pas avoir d'overfitting siginicatif. De plus ces R2 sont très proches de 1, ce qui montre que le modèle capture bien les patterns et les applique correctement..**
<br><br> **-> Les RMSE à 90k$ (train) et 126k$ (test) par semaine semblent énorme. Cepenant, les ventes hébdomadaires de walmart sont dans les millions de dollars. 126k$ sur 1,26M$ c'est environ 8%, et 7% pour 90k$, ce qui est acceptable (au vu du contexte métier).**

### **13 - Analyse des coefficients**
***—> permet d'identifier les features les plus influentes sur les ventes hebdomadaires et d'interpréter le sens de leur impact (positif ou négatif)***


In [19]:
##Création d'un DataFrame pour visualiser et trier les coefficients 
## du modèle par ordre d'importance (valeur absolue décroissante).

#Récupération des noms de features
feature_names = model["preprocessor"].get_feature_names_out()

coefficients = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": model["regressor"].coef_
})

coefficients["Abs_Coefficient"] = coefficients["Coefficient"].abs()
coefficients = coefficients.sort_values("Abs_Coefficient", ascending=False)

print(coefficients)

                  Feature   Coefficient  Abs_Coefficient
2          cat__Store_4.0  1.764386e+06     1.764386e+06
8         cat__Store_10.0  1.681042e+06     1.681042e+06
10        cat__Store_13.0  1.590794e+06     1.590794e+06
3          cat__Store_5.0 -1.398360e+06     1.398360e+06
1          cat__Store_3.0 -1.291490e+06     1.291490e+06
7          cat__Store_9.0 -1.204513e+06     1.204513e+06
16        cat__Store_19.0  9.874872e+05     9.874872e+05
11        cat__Store_14.0  9.718240e+05     9.718240e+05
6          cat__Store_8.0 -8.231173e+05     8.231173e+05
13        cat__Store_16.0 -8.065551e+05     8.065551e+05
5          cat__Store_7.0 -7.048939e+05     7.048939e+05
15        cat__Store_18.0  6.266653e+05     6.266653e+05
22               num__CPI  5.383806e+05     5.383806e+05
17        cat__Store_20.0  4.092194e+05     4.092194e+05
14        cat__Store_17.0  3.076408e+05     3.076408e+05
12        cat__Store_15.0  3.030174e+05     3.030174e+05
0          cat__Store_2.0  2.43

In [20]:
fig = px.bar(coefficients,
             x="Feature",
             y="Coefficient",
             title="Coefficients du modèle baseline par ordre d'importance",
             color="Coefficient",
             color_continuous_scale="RdBu")
fig.show()

**Observations :**

- Les **stores** sont les features les plus influentes sur les ventes hebdomadaires : certains magasins vendent structurellement plus que d'autres (Store_4, Store_10, Store_13).

- **CPI** est la variable économique la plus impactante : l'indice des prix 
à la consommation influence significativement les ventes.

- **Fuel_Price** influence négativement les ventes : quand le prix du carburant 
augmente, les ventes baissent.

- **DayOfWeek a un coefficient de 0** : le jour de la semaine n'a aucune influence, 
ce qui est cohérent car on travaille sur des ventes agrégées à la semaine.

- **Year** a un très faible coefficient : le temps en lui-même n'explique pas 
directement les ventes.

___
___

# **PARTIE III - Ridge ou Lasso + comparaison avec le baseline**

**Ici, nous allons utiliser un model de régression linéaire dit "régularisé". <br> <br> -> Là ou le modèle de régression normal va ajuster les coefficients pour minimiser l'erreur quand il cherchera la meilleure droite de prédiction (avec un risque de donner des coefficients trop élevés à certaines features pour coller parfaitement au train), le modèle régularisé va ajouter des pénalités de façon à ce qu'il reste plus raisonnable et qu'il généralise mieux.**
<br><br>
**Il en existe deux types :<br><br>Ridge → réduit les coefficients sans les mettre à zéro<br>Lasso → peut mettre certains coefficients à zéro → sélection automatique de features**

### **14 - Ridge**
***—> permet d'identifier les features les plus influentes sur les ventes hebdomadaires et d'interpréter le sens de leur impact (positif ou négatif)***


##### **- Entraînement**

In [21]:
#Ridge, avec un alpha à 1

from sklearn.linear_model import Ridge

model_ridge = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=1))
])

model_ridge.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Store', 'Holiday_Flag']),
                                                 ('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Temperature', 'Fuel_Price',
                                                   'CPI', 'Unemployment',
                                                   'Year', 'Month', 'Day',
                                                   'DayOfWeek'])])),
                ('regressor', Ridge(alpha=1))])

##### **- Évaluation (R², RMSE)**

In [22]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np 

#TRAIN
#Prédiction sur données de train
y_pred = model_ridge.predict(X_train)

#Comparaison des prédicitons avec les vraies valeurs
r2_train = r2_score(y_train, y_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred))


#TEST

#Prédiction sur données de test
y_pred = model_ridge.predict(X_test)

#Comparaison des prédicitons avec les vraies valeurs
r2_test = r2_score(y_test, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))


print(f"Train - R²: {r2_train:.4f} | RMSE : {rmse_train:.2f} ")
print(f"Test - R² : {r2_test:.4f} | RMSE : {rmse_test:.2f} ")

Train - R²: 0.9164 | RMSE : 194814.33 
Test - R² : 0.8187 | RMSE : 290905.49 


In [23]:
for alpha in [0.01, 0.1, 1]:
    model_ridge = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", Ridge(alpha=alpha))
    ])
    model_ridge.fit(X_train, y_train)
    y_pred = model_ridge.predict(X_train)
    r2_train = r2_score(y_train, y_pred)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred))
    print(f"Alpha {alpha} → Train R²: {r2_train:.4f} | RMSE: {rmse_train:.2f}")

Alpha 0.01 → Train R²: 0.9819 | RMSE: 90524.11
Alpha 0.1 → Train R²: 0.9787 | RMSE: 98438.14
Alpha 1 → Train R²: 0.9164 | RMSE: 194814.33


##### **- Analyse des coefficients**

In [24]:
coefficients = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": model_ridge["regressor"].coef_
})

coefficients["Abs_Coefficient"] = coefficients["Coefficient"].abs()
coefficients = coefficients.sort_values("Abs_Coefficient", ascending=False)

fig = px.bar(coefficients,
             x="Feature",
             y="Coefficient",
             title="Coefficients Ridge par ordre d'importance",
             color="Coefficient",
             color_continuous_scale="RdBu")
fig.show()

**Observations :**
<br> Les coefficients Ridge sont globalement plus faibles que ceux du modèle baseline —  la régularisation a bien joué son rôle en pénalisant les valeurs extrêmes. 
<br>Cependant, avec alpha=1, les performances sont inférieures au baseline. 
<br> ***Nous allons donc tester plusieurs valeurs d'alpha pour trouver le meilleur équilibre.***

In [25]:
#Comparaison avec des alpha différents

for alpha in [0.01, 0.1, 1]:
    model_ridge = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", Ridge(alpha=alpha))
    ])
    model_ridge.fit(X_train, y_train)
    y_pred = model_ridge.predict(X_train)
    r2_train = r2_score(y_train, y_pred)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred))
    y_pred_test = model_ridge.predict(X_test)
    r2_test = r2_score(y_test, y_pred_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
    
    print(f"Alpha {alpha} → Train : R²: {r2_train:.4f} | RMSE: {rmse_train:.2f}")
    print(f"Alpha {alpha} → Test : R²: {r2_test:.4f} | RMSE: {rmse_test:.2f}")

Alpha 0.01 → Train : R²: 0.9819 | RMSE: 90524.11
Alpha 0.01 → Test : R²: 0.9617 | RMSE: 133699.52
Alpha 0.1 → Train : R²: 0.9787 | RMSE: 98438.14
Alpha 0.1 → Test : R²: 0.9573 | RMSE: 141215.01
Alpha 1 → Train : R²: 0.9164 | RMSE: 194814.33
Alpha 1 → Test : R²: 0.8187 | RMSE: 290905.49


**Observations :**
<br> -> Alpha 0.01 donne les meilleures performances : R² test 0.9617 | RMSE 133k$
<br>-> Cependant Ridge reste légèrement inférieur au baseline 
   (LinearRegression : R² 0.9655 | RMSE 126k$)
<br> -> **Ridge n'améliore pas significativement les performances**

---

### **15 - Lasso**

#### **- Entraînement**

In [26]:
#Lasso, avec un alpha à 1

from sklearn.linear_model import Lasso

model_lasso = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", Lasso(alpha=1, max_iter=100000)) #pour aller plus loin que la recherche de coefficient
])

model_lasso.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Store', 'Holiday_Flag']),
                                                 ('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Temperature', 'Fuel_Price',
                                                   'CPI', 'Unemployment',
                                                   'Year', 'Month', 'Day',
                                                   'DayOfWeek'])])),
                ('regressor', Lasso(alpha=1, max_iter=100000))])

#### **- Évaluation (R², RMSE)**

In [27]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np 

#TRAIN
#Prédiction sur données de train
y_pred = model_lasso.predict(X_train)

#Comparaison des prédicitons avec les vraies valeurs
r2_train = r2_score(y_train, y_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred))


#TEST

#Prédiction sur données de test
y_pred = model_lasso.predict(X_test)

#Comparaison des prédicitons avec les vraies valeurs
r2_test = r2_score(y_test, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))


print(f"Train - R²: {r2_train:.4f} | RMSE : {rmse_train:.2f} ")
print(f"Test - R² : {r2_test:.4f} | RMSE : {rmse_test:.2f} ")

Train - R²: 0.9822 | RMSE : 89799.42 
Test - R² : 0.9652 | RMSE : 127435.67 


**Observations :**
<br><br> Avec un R² et un RMSE proche du model linéaire de base (Train: 0.9822 | Test: 0.9655 | RMSE: 126 960$), on constate que le model Lasso performe quasiment comme ce dernier avec un alpha à 1, quand Ridge lui donnait un R² test à seulement 0.82.
<br><br> ***→ Lasso confirme donc les résultats de la baseline avec une grande stabilité, ce qui suggèe que le dataset ne semble pas contenir de features parasites (à l'exception de DayOfWeek qui est à zéro sur les deux modèles).***

#### **- Analyse des coefficients**

In [28]:
coefficients = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": model_lasso["regressor"].coef_
})

coefficients["Abs_Coefficient"] = coefficients["Coefficient"].abs()
coefficients = coefficients.sort_values("Abs_Coefficient", ascending=False)


fig = px.bar(coefficients,
             x="Feature",
             y="Coefficient",
             title="Coefficients Lasso par ordre d'importance",
             color="Coefficient",
             color_continuous_scale="RdBu")
fig.show()

In [29]:
#Comparaison avec des alpha différents

for alpha in [0.01, 0.1, 1]:
    model_lasso = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", Lasso(alpha=alpha, max_iter= 100000))
    ])
    model_lasso.fit(X_train, y_train)
    y_pred = model_lasso.predict(X_train)
    r2_train = r2_score(y_train, y_pred)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred))
    y_pred_test = model_lasso.predict(X_test)
    r2_test = r2_score(y_test, y_pred_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
    
    print(f"Alpha {alpha} → Train : R²: {r2_train:.4f} | RMSE: {rmse_train:.2f}")
    print(f"Alpha {alpha} → Test : R²: {r2_test:.4f} | RMSE: {rmse_test:.2f}")

Alpha 0.01 → Train : R²: 0.9822 | RMSE: 89796.67
Alpha 0.01 → Test : R²: 0.9655 | RMSE: 126965.25
Alpha 0.1 → Train : R²: 0.9822 | RMSE: 89796.70
Alpha 0.1 → Test : R²: 0.9654 | RMSE: 127007.67
Alpha 1 → Train : R²: 0.9822 | RMSE: 89799.42
Alpha 1 → Test : R²: 0.9652 | RMSE: 127435.67


**Observations :**
<br><br> ***→ Cela nous confirme l'observation post-évaluation. Même avec des pénalisations différentes, Lasso, étant pourtant plus radical que Ridge, reste toujours très stable. Cela appuie l'hypothèse selon laquelle le dataset est très propre et que ses features sonttoutes pertinentes (à l'exception de DayOfWeek qui est à zéro sur les deux modèles).***

___
___
# **PARTIE IV - CONCLUSION**

### **16 - Comparaison Baseline vs Ridge vs Lasso**

In [30]:
comparison_df = pd.DataFrame({
    "Modèle": ["Baseline", "Ridge", "Lasso"],
    "Train R²": [0.9822, 0.9819, 0.9822],
    "Test R²": [0.9655, 0.9617, 0.9655],
    "Train RMSE":[89796, 90524, 89796],
    "Test RMSE": [126960, 133699, 126965]
})

print(comparison_df)

fig = px.bar(comparison_df,
             x="Modèle",
             y=["Train R²", "Test R²"],
             title="Comparaison R² des modèles",
             barmode="group")
fig.show()

fig2 = px.bar(comparison_df,
              x="Modèle",
              y=["Train RMSE", "Test RMSE"],
              title="Comparaison RMSE des modèles",
              barmode="group")
fig2.show()

     Modèle  Train R²  Test R²  Train RMSE  Test RMSE
0  Baseline    0.9822   0.9655       89796     126960
1     Ridge    0.9819   0.9617       90524     133699
2     Lasso    0.9822   0.9655       89796     126965


### **17 - Conclusion et recommandations**

*En analysant le récapitulatif, on peut observer des **résultats très proches entre le modèle baseline et Lasso** avec un RMSE différent que de 5$.<br>
On pourrait choisir le modèle de base, bien que les 5$ de plus semblent négrligeable, cependant, il est **préférable de choisir le modèle avec Lasso (alpha = 0.01). <br><br>***
**En effet, pour 5$, nous nous retrouvons avec un modèle avec des performances identiques, mais :**<br>
**→ une meilleure robustesse,**<br>
**→ le tri des features inutiles (ce qui serait utilie en cas d'ajout de nouvelles features au dataset).**

#### **→ A performance égale, nous préférerons le modèle régularisé.**

___
___
___
## **PARTIE BONUS**

### **18 - Gridsearch**

#### Gridseach permet d'automatiser l'étape de test des alphas:
**-> il va directement tester la liste d'alphas qu'on lui donne et retourner le meilleur,<br>**
**-> avec sa Cross Validation il sépare l'échantillon train en plusieurs parties et teste le modèle avec ces combinaisons différentes**

In [31]:
from sklearn.model_selection import GridSearchCV

# Liste des alphas à tester
param_grid = {"regressor__alpha": [0.001, 0.01, 0.1, 1, 10, 100]}

# Pipeline validée : Lasso
pipeline_lasso = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", Lasso(max_iter=100000))
])

# GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline_lasso,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error"
)

import warnings
warnings.filterwarnings("ignore")
grid_search.fit(X_train, y_train)

print(f"Meilleur alpha : {grid_search.best_params_}")
print(f"Meilleur score : {grid_search.best_score_:.4f}")


Meilleur alpha : {'regressor__alpha': 1}
Meilleur score : -50130258306.1108


Les warnings semblent indiquer que lors de certains folds de cross-validation, 
des stores présents dans le fold de validation n'apparaissent pas dans le fold 
d'entraînement — le OneHotEncoder les encode alors à zéro. 
Cela est dû à la petite taille du dataset (80 lignes) et n'impacte pas 
la validité des résultats.

In [32]:
best_model = grid_search.best_estimator_

y_pred_test = best_model.predict(X_test)
r2_test = r2_score(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print(f"Meilleur modèle - Test R²: {r2_test:.4f} | RMSE: {rmse_test:.2f}")

Meilleur modèle - Test R²: 0.9652 | RMSE: 127435.67


### **→ La différence avec Lasso en manuel est quasi nulle. GridSearch nous confirme à nouveau notre conclusion sur la propreté du dataset et sa sensibilité moindre face à l'hyperparamètre alpha.**